In [1]:
# Install required packages
!pip install -q rank-bm25 sentence-transformers langchain-google-genai langchain-groq langchain python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.6 MB/s eta 0:00:00


In [3]:
# Install required packages - RUN THIS CELL FIRST
!pip install -q rank-bm25 sentence-transformers langchain-google-genai langchain-groq langchain langchain-core langchain-community python-dotenv

print("Installation complete! Please restart the runtime now:")
print("Runtime → Restart runtime (Ctrl+M)")
print("Then run the remaining cells.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
Installation complete! Please restart the runtime now:
Runtime → Restart runtime (Ctrl+M)
Then run the remaining cells.


In [22]:
import numpy as np
import os
import sys
from typing import List, Dict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# LangChain imports (compatible with latest versions)
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_groq import ChatGroq
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import StrOutputParser
except ImportError as e:
    print(f"Import error: {e}")
    print("Please restart the runtime (Runtime → Restart runtime) and try again.")
    raise

from google.colab import userdata
import textwrap

# Setup API keys (set these in Colab Secrets or replace with your keys)
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini-API")
    os.environ["GROQ_API_KEY"] = userdata.get("Groq")
    print("API keys loaded from Colab secrets.")
except:
    print("Warning: Could not load API keys from secrets.")
    print("Please set GOOGLE_API_KEY and GROQ_API_KEY manually or in environment variables.")
    # Fallback - replace with your keys if not using secrets:
    # os.environ["GOOGLE_API_KEY"] = "your-key-here"
    # os.environ["GROQ_API_KEY"] = "your-key-here"

API keys loaded from Colab secrets.


In [23]:
# ============================================================
# PART 1: Document Corpus (10+ documents)
# ============================================================

corpus = [
    # Neural network training sub-topic 1: Optimization Algorithms
    "Stochastic Gradient Descent (SGD) updates parameters using mini-batch approximations of the full gradient, reducing computational overhead per iteration.",
    "Adam optimizer combines momentum and adaptive learning rates, computing first and second moment estimates of gradients for faster convergence.",

    # Neural network training sub-topic 2: Backpropagation & Gradients
    "Backpropagation applies the chain rule to compute gradients of the loss function with respect to each weight in the network.",
    "The vanishing gradient problem occurs when gradients become exponentially small in deep networks using sigmoid or tanh activation functions.",

    # Neural network training sub-topic 3: Regularization Techniques
    "Dropout randomly sets neurons to zero during training with probability p, preventing co-adaptation and reducing overfitting in deep neural networks.",
    "L2 regularization adds a penalty term proportional to the squared magnitude of weights, encouraging smaller weight values and simpler models.",

    # Technical jargon document (BM25 should excel here - contains specific technical terms)
    "The CUDA kernel launches thousands of threads organized in blocks and grids to parallelize matrix multiplication operations on NVIDIA GPU tensor cores.",

    # Transformer architecture documents
    "Transformers use self-attention mechanisms to process sequences in parallel, computing scaled dot-product attention scores between all token pairs.",
    "BERT is a bidirectional transformer encoder trained using masked language modelling to learn deep contextualized word representations.",

    # Other AI/ML topics
    "Support Vector Machines construct optimal hyperplanes in high-dimensional feature space to maximize the margin between different class boundaries.",
    "Random Forest ensembles combine multiple decision trees using bootstrap aggregating and random feature selection to reduce variance.",
    "The REINFORCE algorithm estimates policy gradients using Monte Carlo sampling to optimize reinforcement learning agents for sequential decision tasks."
]

print(f"Corpus contains {len(corpus)} documents")
print("\nSample documents:")
for i, doc in enumerate(corpus[:3]):
    print(f"  [{i}]: {doc[:80]}...")

Corpus contains 12 documents

Sample documents:
  [0]: Stochastic Gradient Descent (SGD) updates parameters using mini-batch approximat...
  [1]: Adam optimizer combines momentum and adaptive learning rates, computing first an...
  [2]: Backpropagation applies the chain rule to compute gradients of the loss function...


In [24]:
# ============================================================
# PART 2: Hybrid Retriever with RRF
# ============================================================

class HybridRetriever:
    def __init__(self, corpus: List[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # Initialize BM25 (Sparse Retrieval)
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # Initialize SBERT (Dense Retrieval)
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        print("Encoding corpus with SBERT...")
        self.corpus_embeddings = self.sbert_model.encode(corpus, convert_to_tensor=False)
        print("Encoding complete.")

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        # ---- BM25 Scores and Ranks ----
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

        # ---- SBERT Scores and Ranks ----
        query_embedding = self.sbert_model.encode(query, convert_to_tensor=False)
        # Cosine similarity
        sbert_scores = np.dot(self.corpus_embeddings, query_embedding) / (
            np.linalg.norm(self.corpus_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

        # ---- Reciprocal Rank Fusion (RRF) ----
        # RRF(d) = sum over retrievers: 1 / (k + rank_retriever(d))
        results = []
        for doc_id in range(len(self.corpus)):
            bm25_rank = bm25_ranks[doc_id]
            sbert_rank = sbert_ranks[doc_id]

            rrf_score = (1 / (self.k + bm25_rank)) + (1 / (self.k + sbert_rank))

            results.append({
                "doc_id": doc_id,
                "rrf_score": rrf_score,
                "bm25_rank": bm25_rank,
                "sbert_rank": sbert_rank,
                "text": self.corpus[doc_id],
                "bm25_score": bm25_scores[doc_id],
                "sbert_score": sbert_scores[doc_id]
            })

        # Sort by RRF score descending
        results.sort(key=lambda x: x["rrf_score"], reverse=True)
        return results[:top_k]


# Demonstration of Hybrid Retrieval
retriever = HybridRetriever(corpus)
query = "optimization techniques for training"

print(f"\nQuery: '{query}'")
print("="*80)
hybrid_results = retriever.retrieve(query, top_k=3)

print(f"{'Doc ID':<8} {'RRF Score':<12} {'BM25 Rank':<12} {'SBERT Rank':<12} {'Text':<40}")
print("-"*80)
for res in hybrid_results:
    short_text = res['text'][:37] + "..."
    print(f"{res['doc_id']:<8} {res['rrf_score']:<12.4f} {res['bm25_rank']:<12} {res['sbert_rank']:<12} {short_text}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding corpus with SBERT...
Encoding complete.

Query: 'optimization techniques for training'
Doc ID   RRF Score    BM25 Rank    SBERT Rank   Text                                    
--------------------------------------------------------------------------------
1        0.0323       3            1            Adam optimizer combines momentum and ...
11       0.0318       2            4            The REINFORCE algorithm estimates pol...
4        0.0313       1            7            Dropout randomly sets neurons to zero...


In [25]:
# ============================================================
# PART 3: Cross-Encoder Re-Ranker
# ============================================================

class CrossEncoderReranker:
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        print(f"Loading Cross-Encoder: {model_name}...")
        self.model = CrossEncoder(model_name)
        print("Cross-Encoder loaded.")

    def rerank(self, query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
        """
        Re-ranks candidates using the original user query (not expanded).
        Returns top_k documents with cross-encoder scores.
        """
        if not candidates:
            return []

        # Prepare query-document pairs
        pairs = [(query, candidate["text"]) for candidate in candidates]

        # Get cross-encoder scores (higher = more relevant)
        scores = self.model.predict(pairs)

        # Add scores to candidate dicts
        reranked_candidates = []
        for i, candidate in enumerate(candidates):
            new_candidate = candidate.copy()
            new_candidate["cross_encoder_score"] = float(scores[i])
            reranked_candidates.append(new_candidate)

        # Sort by cross-encoder score descending
        reranked_candidates.sort(key=lambda x: x["cross_encoder_score"], reverse=True)

        return reranked_candidates[:top_k]


# Demonstration of Re-ranking
reranker = CrossEncoderReranker()

# Get more candidates from hybrid retrieval, then re-rank
candidates = retriever.retrieve(query, top_k=10)
reranked = reranker.rerank(query, candidates, top_k=3)

print(f"\nAfter Cross-Encoder Re-ranking (Original query: '{query}'):")
print(f"{'Doc ID':<8} {'CE Score':<12} {'Previous RRF':<15} {'Text':<40}")
print("-"*80)
for res in reranked:
    short_text = res['text'][:37] + "..."
    print(f"{res['doc_id']:<8} {res['cross_encoder_score']:<12.4f} {res['rrf_score']:<15.4f} {short_text}")

Loading Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder loaded.

After Cross-Encoder Re-ranking (Original query: 'optimization techniques for training'):
Doc ID   CE Score     Previous RRF    Text                                    
--------------------------------------------------------------------------------
11       -3.6190      0.0318          The REINFORCE algorithm estimates pol...
4        -4.7279      0.0313          Dropout randomly sets neurons to zero...
1        -7.7268      0.0323          Adam optimizer combines momentum and ...


In [26]:
# ============================================================
# PART 4: Query Expansion using HyDE (Hypothetical Document Embedding)
# ============================================================

class HyDEQueryExpander:
    def __init__(self):
        # Initialize Gemini with low temperature for deterministic hypothetical answers
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash",
            temperature=0.0  # Deterministic output for HyDE
        )

        self.prompt = PromptTemplate(
            input_variables=["query"],
            template="""You are an expert AI/ML professor. Given the student question below, generate a hypothetical textbook passage that would answer this question.
The passage should be detailed, technical, and contain specific terminology that would appear in academic literature (2-3 sentences).

Student Question: {query}

Hypothetical Answer Passage:"""
        )

        self.chain = self.prompt | self.llm | StrOutputParser()

    def expand(self, query: str) -> str:
        """Generate hypothetical document for retrieval"""
        hypothetical_doc = self.chain.invoke({"query": query})
        return hypothetical_doc.strip()


# Demonstration of HyDE
expander = HyDEQueryExpander()

test_query = "what is attention?"
expanded = expander.expand(test_query)

print(f"Original Query: {test_query}")
print(f"\nHyDE Expanded Document:\n{textwrap.fill(expanded, width=80)}")

# Show retrieval difference
print(f"\n--- Retrieval Comparison ---")
print(f"Without HyDE (direct):")
direct_results = retriever.retrieve(test_query, top_k=1)
print(f"  Top doc: [{direct_results[0]['doc_id']}] {direct_results[0]['text'][:60]}...")

print(f"\nWith HyDE (expanded):")
hyde_results = retriever.retrieve(expanded, top_k=1)
print(f"  Top doc: [{hyde_results[0]['doc_id']}] {hyde_results[0]['text'][:60]}...")

Original Query: what is attention?

HyDE Expanded Document:
Attention is a neural mechanism that allows a model to dynamically weigh the
importance of different input elements when processing information or generating
an output, thereby overcoming the representational bottleneck of fixed-size
vectors. This is typically achieved by computing scalar attention scores, often
derived from compatibility functions (e.g., dot products) between a query vector
and various key vectors representing input elements, which are then normalized
(e.g., via softmax) to form a probability distribution. These normalized scores
subsequently serve as weights for a weighted sum of corresponding value vectors,
yielding a context vector that adaptively focuses on the most salient
information for the task at hand, a fundamental component in modern sequence-to-
sequence models and large language models.

--- Retrieval Comparison ---
Without HyDE (direct):
  Top doc: [8] BERT is a bidirectional transformer encoder

In [27]:
# ============================================================
# PART 5: End-to-End Advanced RAG Pipeline
# ============================================================

class AdvancedRAGSystem:
    def __init__(self, corpus: List[str]):
        self.corpus = corpus
        self.retriever = HybridRetriever(corpus)
        self.reranker = CrossEncoderReranker()
        self.expander = HyDEQueryExpander()

        # LLM for final generation (using Groq for speed)
        self.llm = ChatGroq(
            model_name="llama-3.1-8b-instant",
            temperature=0.3
        )

        self.rag_prompt = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant for a university AI/ML course. Use the following retrieved context to answer the student's question.
If the context doesn't contain the answer, say "I don't have enough information to answer this question."

Retrieved Context:
{context}

Student Question: {question}

Provide a clear, accurate answer:"""
        )

        self.chain = self.rag_prompt | self.llm | StrOutputParser()

    def advanced_rag(self, user_query: str, verbose: bool = True) -> str:
        """Full pipeline: HyDE → Hybrid Retrieval → Re-Ranking → Generation"""
        if verbose:
            print(f"\n{'='*80}")
            print(f"QUERY: {user_query}")
            print(f"{'='*80}")

        # Step 1: Query Expansion (HyDE)
        expanded_query = self.expander.expand(user_query)
        if verbose:
            print(f"\n[1] HyDE Expansion:")
            print(f"    {textwrap.fill(expanded_query, width=75, initial_indent='    ', subsequent_indent='    ')}")

        # Step 2: Hybrid Retrieval (using expanded query)
        initial_results = self.retriever.retrieve(expanded_query, top_k=10)
        if verbose:
            print(f"\n[2] Hybrid Retrieval (top 5 of 10):")
            for i, res in enumerate(initial_results[:5]):
                print(f"    [{res['doc_id']}] RRF:{res['rrf_score']:.4f} | {res['text'][:50]}...")

        # Step 3: Re-Ranking (using original query)
        reranked_results = self.reranker.rerank(user_query, initial_results, top_k=3)
        if verbose:
            print(f"\n[3] After Cross-Encoder Re-ranking:")
            for i, res in enumerate(reranked_results):
                print(f"    [{res['doc_id']}] CE:{res['cross_encoder_score']:.4f} | {res['text'][:50]}...")

        # Step 4: Generation
        context = "\n\n".join([f"[{i+1}] {doc['text']}" for i, doc in enumerate(reranked_results)])
        answer = self.chain.invoke({"context": context, "question": user_query})

        if verbose:
            print(f"\n[4] Generated Answer:")
            print(f"    {textwrap.fill(answer, width=75, initial_indent='    ', subsequent_indent='    ')}")

        return answer, reranked_results

    def naive_rag(self, user_query: str) -> tuple:
        """
        Naïve RAG: Dense-only (SBERT), no expansion, no re-ranking.
        Returns (answer, top_doc_id)
        """
        # Direct SBERT retrieval (cosine similarity only)
        query_embedding = self.retriever.sbert_model.encode(user_query, convert_to_tensor=False)
        scores = np.dot(self.retriever.corpus_embeddings, query_embedding) / (
            np.linalg.norm(self.retriever.corpus_embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        top_indices = np.argsort(scores)[::-1][:3]

        context = "\n\n".join([f"[{i+1}] {self.corpus[idx]}" for i, idx in enumerate(top_indices)])
        answer = self.chain.invoke({"context": context, "question": user_query})

        return answer, top_indices[0], self.corpus[top_indices[0]]


# Initialize the system
print("Initializing Advanced RAG System...")
rag_system = AdvancedRAGSystem(corpus)
print("System ready!")

Initializing Advanced RAG System...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding corpus with SBERT...
Encoding complete.
Loading Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-Encoder loaded.
System ready!


In [28]:
# ============================================================
# PART 6: Comparison Experiment (Naïve vs Advanced RAG)
# ============================================================

test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "how to prevent overfitting in neural networks?"
]

print("RUNNING COMPARISON EXPERIMENT")
print("="*100)
print("Naïve RAG  = Dense-only (SBERT cosine), no expansion, no re-ranking")
print("Advanced RAG = Rule-Based Expansion → Hybrid (BM25+SBERT+RRF) → Cross-Encoder")
print("="*100)

results_data = []

for q in test_queries:
    print(f"\n\nQuery: '{q}'")
    print("-"*100)

    # Naïve RAG
    naive_answer, naive_top_id, naive_top_doc = rag_system.naive_rag(q)

    # Advanced RAG
    advanced_answer, advanced_docs = rag_system.advanced_rag(q, verbose=False)
    advanced_top_doc = advanced_docs[0]['text'] if advanced_docs else "None"
    advanced_top_id = advanced_docs[0]['doc_id'] if advanced_docs else "N/A"

    # Store results
    results_data.append({
        'query': q,
        'naive_top': f"[{naive_top_id}] {naive_top_doc[:50]}...",
        'advanced_top': f"[{advanced_top_id}] {advanced_top_doc[:50]}...",
        'different': naive_top_doc != advanced_top_doc
    })

    # Display
    print(f"\nNaïve RAG Top Doc:    [{naive_top_id}] {naive_top_doc[:70]}...")
    print(f"Advanced RAG Top Doc: [{advanced_top_id}] {advanced_top_doc[:70]}...")
    print(f"Documents Different?: {'YES' if naive_top_doc != advanced_top_doc else 'NO'}")

# Final markdown table
print("\n\n" + "="*100)
print("FINAL COMPARISON TABLE (Markdown format)")
print("="*100)
print(f"\n| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |")
print(f"|---|---|---|---|")

for r in results_data:
    query_clean = r['query'].replace('|', '\\|')
    naive_clean = r['naive_top'].replace('|', '\\|')
    advanced_clean = r['advanced_top'].replace('|', '\\|')
    diff_clean = "Yes" if r['different'] else "No"
    print(f"| `{query_clean}` | {naive_clean} | {advanced_clean} | {diff_clean} |")

RUNNING COMPARISON EXPERIMENT
Naïve RAG  = Dense-only (SBERT cosine), no expansion, no re-ranking
Advanced RAG = Rule-Based Expansion → Hybrid (BM25+SBERT+RRF) → Cross-Encoder


Query: 'how do transformers encode meaning?'
----------------------------------------------------------------------------------------------------

Naïve RAG Top Doc:    [7] Transformers use self-attention mechanisms to process sequences in par...
Advanced RAG Top Doc: [7] Transformers use self-attention mechanisms to process sequences in par...
Documents Different?: NO


Query: 'optimization techniques for training'
----------------------------------------------------------------------------------------------------

Naïve RAG Top Doc:    [1] Adam optimizer combines momentum and adaptive learning rates, computin...
Advanced RAG Top Doc: [11] The REINFORCE algorithm estimates policy gradients using Monte Carlo s...
Documents Different?: YES


Query: 'how to prevent overfitting in neural networks?'
---------------

In [29]:
# ============================================================
# NUMERICAL WALKTHROUGH: BM25 + SBERT + RRF (Similar to assignment example)
# Query: "how do neural nets learn?"
# ============================================================

demo_query = "how do neural nets learn?"
print(f"Query: '{demo_query}'")
print("="*80)

# Step 1: BM25 Scores
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(demo_query.lower().split())

print("\nSTEP 1: BM25 Raw Scores")
print("-"*80)
for i, s in enumerate(bm25_scores):
    marker = " <--" if s == max(bm25_scores) else ""
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:55]}{marker}")

bm25_ranked = np.argsort(bm25_scores)[::-1]
bm25_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}
print(f"\nBM25 Ranking (best→worst): {list(bm25_ranked)}")

# Step 2: SBERT Scores
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
query_emb = sbert_model.encode(demo_query)
corpus_emb = sbert_model.encode(corpus)
sbert_scores = np.dot(corpus_emb, query_emb) / (np.linalg.norm(corpus_emb, axis=1) * np.linalg.norm(query_emb))

print("\nSTEP 2: SBERT Cosine Similarities")
print("-"*80)
for i, s in enumerate(sbert_scores):
    marker = " <--" if s == max(sbert_scores) else ""
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:55]}{marker}")

sbert_ranked = np.argsort(sbert_scores)[::-1]
sbert_ranks = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}
print(f"\nSBERT Ranking (best→worst): {list(sbert_ranked)}")

# Step 3: RRF Calculation
k = 60
print(f"\nSTEP 3: RRF Scoring (k={k})")
print(f"Formula: RRF(d) = 1/({k}+r_bm25) + 1/({k}+r_sbert)")
print("-"*80)
rrf_results = []
for doc_id in range(len(corpus)):
    rrf_score = (1/(k + bm25_ranks[doc_id])) + (1/(k + sbert_ranks[doc_id]))
    rrf_results.append((doc_id, rrf_score, bm25_ranks[doc_id], sbert_ranks[doc_id]))
    print(f"  doc_{doc_id}: 1/({k}+{bm25_ranks[doc_id]}) + 1/({k}+{sbert_ranks[doc_id]}) = {rrf_score:.5f}")

rrf_results.sort(key=lambda x: x[1], reverse=True)
print(f"\nFinal RRF Ranking (best→worst): {[x[0] for x in rrf_results]}")
print(f"\nWinner: Document {rrf_results[0][0]} with score {rrf_results[0][1]:.5f}")
print(f"Text: {corpus[rrf_results[0][0]]}")

Query: 'how do neural nets learn?'

STEP 1: BM25 Raw Scores
--------------------------------------------------------------------------------
  doc_0: 0.0000  |  Stochastic Gradient Descent (SGD) updates parameters us
  doc_1: 0.0000  |  Adam optimizer combines momentum and adaptive learning 
  doc_2: 0.0000  |  Backpropagation applies the chain rule to compute gradi
  doc_3: 0.0000  |  The vanishing gradient problem occurs when gradients be
  doc_4: 1.9857  |  Dropout randomly sets neurons to zero during training w <--
  doc_5: 0.0000  |  L2 regularization adds a penalty term proportional to t
  doc_6: 0.0000  |  The CUDA kernel launches thousands of threads organized
  doc_7: 0.0000  |  Transformers use self-attention mechanisms to process s
  doc_8: 0.0000  |  BERT is a bidirectional transformer encoder trained usi
  doc_9: 0.0000  |  Support Vector Machines construct optimal hyperplanes i
  doc_10: 0.0000  |  Random Forest ensembles combine multiple decision trees
  doc_11: 0.0000  

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



STEP 2: SBERT Cosine Similarities
--------------------------------------------------------------------------------
  doc_0: 0.2729  |  Stochastic Gradient Descent (SGD) updates parameters us
  doc_1: 0.4316  |  Adam optimizer combines momentum and adaptive learning  <--
  doc_2: 0.4269  |  Backpropagation applies the chain rule to compute gradi
  doc_3: 0.3476  |  The vanishing gradient problem occurs when gradients be
  doc_4: 0.3519  |  Dropout randomly sets neurons to zero during training w
  doc_5: 0.1869  |  L2 regularization adds a penalty term proportional to t
  doc_6: 0.1115  |  The CUDA kernel launches thousands of threads organized
  doc_7: 0.2476  |  Transformers use self-attention mechanisms to process s
  doc_8: 0.2833  |  BERT is a bidirectional transformer encoder trained usi
  doc_9: 0.2409  |  Support Vector Machines construct optimal hyperplanes i
  doc_10: 0.0053  |  Random Forest ensembles combine multiple decision trees
  doc_11: 0.3053  |  The REINFORCE algorith